In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('../data/cleaned_data.csv')

print("--- ADVANCED ANALYSIS ---")

# 1. Chi-Square Test (Contract vs Churn)
# H0: Contract type and Churn are independent
contingency_table = pd.crosstab(df['Contract'], df['Churn'])
chi2, p_val_chi, dof, expected = chi2_contingency(contingency_table)
print(f"\n1. Chi-Square Test (Contract vs Churn):")
print(f"P-value: {p_val_chi:.4e}")
if p_val_chi < 0.05:
    print("Result: Significant relationship found between Contract Type and Churn.")

# 2. T-Test (Monthly Charges: Churned vs Retained)
# H0: No difference in average monthly charges between churned and retained customers
churned_charges = df[df['Churn'] == 1]['MonthlyCharges']
retained_charges = df[df['Churn'] == 0]['MonthlyCharges']
t_stat, p_val_t = ttest_ind(retained_charges, churned_charges, equal_var=False)
print(f"\n2. T-Test (Monthly Charges):")
print(f"P-value: {p_val_t:.4e}")
if p_val_t < 0.05:
    print("Result: Significant difference in monthly charges between the two groups.")

# 3. Machine Learning: Feature Importance using Random Forest
print("\n3. Feature Importance (Random Forest):")
# Prepare data for ML (encode categorical variables)
le = LabelEncoder()
ml_df = df.copy()
categorical_cols = ['Contract', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen_Label', 'Tenure_Group']
for col in categorical_cols:
    ml_df[col] = le.fit_transform(ml_df[col])

# Define features (X) and target (y)
X = ml_df.drop(['CustomerID', 'Churn', 'Churn_Label'], axis=1)
y = ml_df['Churn']

# Train Model
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X, y)

# Plot Feature Importance
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
plt.title('What Drives Customer Churn? (Feature Importance)', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()